In [2]:
import numpy as np

def get_sgp_mat(num_in, num_out, link):
    A = np.zeros((num_in, num_out))
    for i, j in link:
        A[i, j] = 1
    A_norm = A / np.sum(A, axis=0, keepdims=True)
    return A_norm

# 返回的是节点自身的0阶邻居矩阵
def edge2mat(link, num_node):
    A = np.zeros((num_node, num_node))
    for i, j in link:
        A[j, i] = 1
    return A

def get_k_scale_graph(scale, A):
    if scale == 1:
        return A
    An = np.zeros_like(A)
    A_power = np.eye(A.shape[0])
    for k in range(scale):
        A_power = A_power @ A
        An += A_power
    An[An > 0] = 1
    return An

# 邻居矩阵乘以度矩阵的倒数
def normalize_digraph(A):
    Dl = np.sum(A, 0)
    h, w = A.shape
    Dn = np.zeros((w, w))
    for i in range(w):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** (-1)
    AD = np.dot(A, Dn)
    return AD

# inward方向和outward方向都需要做乘以度矩阵
# inward和outward
def get_spatial_graph(num_node, self_link, inward, outward):
    I = edge2mat(self_link, num_node) # 0 阶邻居矩阵
    In = normalize_digraph(edge2mat(inward, num_node)) # In邻接矩阵乘以度矩阵的倒数进行规范化
    Out = normalize_digraph(edge2mat(outward, num_node))
    A = np.stack((I, In, Out)) # A.shape = (3,25,25)
    return A

def normalize_adjacency_matrix(A):
    node_degrees = A.sum(-1)
    degs_inv_sqrt = np.power(node_degrees, -0.5)
    norm_degs_matrix = np.eye(len(node_degrees)) * degs_inv_sqrt
    return (norm_degs_matrix @ A @ norm_degs_matrix).astype(np.float32)


def k_adjacency(A, k, with_self=False, self_factor=1):
    assert isinstance(A, np.ndarray)
    I = np.eye(len(A), dtype=A.dtype)
    if k == 0:
        return I
    Ak = np.minimum(np.linalg.matrix_power(A + I, k), 1) \
       - np.minimum(np.linalg.matrix_power(A + I, k - 1), 1)
    if with_self:
        Ak += (self_factor * I)
    return Ak

def get_multiscale_spatial_graph(num_node, self_link, inward, outward):
    I = edge2mat(self_link, num_node)
    A1 = edge2mat(inward, num_node)
    A2 = edge2mat(outward, num_node)
    A3 = k_adjacency(A1, 2)
    A4 = k_adjacency(A2, 2)
    A1 = normalize_digraph(A1)
    A2 = normalize_digraph(A2)
    A3 = normalize_digraph(A3)
    A4 = normalize_digraph(A4)
    A = np.stack((I, A1, A2, A3, A4))
    return A


num_node = 25

self_link = [(i, i) for i in range(num_node)]

inward = [
    (1, 0), (2, 1), (3, 2), (4, 3),
    (5, 1), (6, 5), (7, 6),
    (8, 1), (9, 8), (10, 9),
    (11, 8), (12, 11), (13, 12),
    (14, 0), (15, 0),
    (16, 14), (17, 15),
    (18, 14), (19, 18), (20, 19),
    (21, 14), (22, 11),
    (23, 22), (24, 11)
]

outward = [(j, i) for (i, j) in inward]

neighbor = inward + outward


def edge2mat(link, num_node):
    A = np.zeros((num_node, num_node))
    for i, j in link:
        A[j, i] = 1
    return A


def normalize_digraph(A):
    Dl = np.sum(A, 0)
    num_node = A.shape[0]
    Dn = np.zeros((num_node, num_node))
    for i in range(num_node):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** (-1)
    AD = np.dot(A, Dn)
    return AD


def get_spatial_graph(num_node, self_link, inward, outward):

    I = edge2mat(self_link, num_node)

    In = normalize_digraph(edge2mat(inward, num_node))

    Out = normalize_digraph(edge2mat(outward, num_node))

    A = np.stack((I, In, Out))

    return A


class Graph:

    def __init__(self, labeling_mode='spatial'):

        self.num_node = num_node
        self.self_link = self_link
        self.inward = inward
        self.outward = outward
        self.neighbor = neighbor

        self.A = self.get_adjacency_matrix(labeling_mode)

    def get_adjacency_matrix(self, labeling_mode=None):

        if labeling_mode is None:
            return self.A

        if labeling_mode == 'spatial':
            A = get_spatial_graph(
                self.num_node,
                self.self_link,
                self.inward,
                self.outward
            )

        else:
            raise ValueError("Unsupported labeling mode")

        return A

In [3]:
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


LEAKY_ALPHA = 0.1
def init_param(modules):
    for m in modules:
        if isinstance(m, nn.Conv1d) or isinstance(m, nn.Conv2d) or isinstance(m, nn.Conv3d):
            nn.init.kaiming_normal_(m.weight, a=LEAKY_ALPHA, mode='fan_out', nonlinearity='leaky_relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm1d) or isinstance(m, nn.BatchNorm2d) or isinstance(m, nn.BatchNorm3d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
            
            
class TemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1, groups=1):
        super(TemporalConv, self).__init__()
        
        pad = (kernel_size + (kernel_size-1) * (dilation-1) - 1) // 2
        self.conv = nn.Conv2d(in_channels, 
                              out_channels, 
                              kernel_size=(kernel_size, 1),
                              padding=(pad, 0), 
                              stride=(stride, 1), 
                              dilation=(dilation, 1), 
                              groups=groups)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.bn(self.conv(x))
        return x
    
    
class PointWiseTCN(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, groups=1):
        super(PointWiseTCN, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 1, stride=(stride, 1), groups=groups)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.bn(self.conv(x))
        return x
    
    
class UnfoldTemporalWindows(nn.Module):
    def __init__(self, window_size, window_stride, window_dilation=1, pad=True):
        super().__init__()
        
        self.window_size = window_size
        self.padding = (window_size + (window_size-1) * (window_dilation-1) - 1) // 2 if pad else 0
        
        self.unfold = nn.Unfold(kernel_size=(self.window_size, 1),
                                dilation=(self.window_dilation, 1),
                                stride=(self.window_stride, 1),
                                padding=(self.padding, 0))

    def forward(self, x):
        N, C, T, V = x.shape
        x = self.unfold(x)
        x = x.view(N, C, self.window_size, -1, V)
        x = x.transpose(2, 3).contiguous()
        return x
    
    
class PositionalEncoding(nn.Module):
    def __init__(self, channel, joint_num, time_len):
        super(PositionalEncoding, self).__init__()
        self.joint_num = joint_num
        self.time_len = time_len

        pos_list = []
        for t in range(self.time_len):
            for j_id in range(self.joint_num):
                pos_list.append(j_id)
        position = torch.from_numpy(np.array(pos_list)).unsqueeze(1).float()

        pe = torch.zeros(self.time_len * self.joint_num, channel)
        div_term = torch.exp(torch.arange(0, channel, 2).float() *
                             -(math.log(10000.0) / channel))  # channel//2
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.view(time_len, joint_num, channel).permute(2, 0, 1).unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):  # nctv
        x = x + self.pe.to(x.dtype)[:, :, :x.size(2)]
        return x   
    
    
class ST_GC(nn.Module):
    def __init__(self, in_channels, out_channels, A):
        super(ST_GC, self).__init__()
        
        A = torch.from_numpy(A.astype(np.float32))
        self.A = nn.Parameter(A)
        self.Nh = A.size(0)
        
        self.conv = nn.Conv2d(in_channels, out_channels * self.Nh, 1)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        N, C, T, V = x.size()
        v = self.conv(x).view(N, self.Nh, -1, T, V)
        weights = self.A.to(v.dtype)
        
        x = torch.einsum('hvu,nhctu->nctv', weights, v)
        x = self.bn(x)
        return x
    

class CTR_GC(nn.Module):
    def __init__(self, in_channels, out_channels, A, num_scale=1):
        super(CTR_GC, self).__init__()

        A = torch.from_numpy(A.astype(np.float32))
        self.Nh = A.size(0)
        self.A = nn.Parameter(A)
        self.num_scale = num_scale
        
        rel_channels = in_channels // 8 if in_channels != 3 else 8
        
        self.conv1 = nn.Conv2d(in_channels, rel_channels * self.Nh, 1, groups=num_scale)
        self.conv2 = nn.Conv2d(in_channels, rel_channels * self.Nh, 1, groups=num_scale)
        self.conv3 = nn.Conv2d(in_channels, out_channels * self.Nh, 1, groups=num_scale)
        self.conv4 = nn.Conv2d(rel_channels * self.Nh, out_channels * self.Nh, 1, groups=num_scale * self.Nh)
        
        self.alpha = nn.Parameter(torch.zeros(1))
        self.bn = nn.BatchNorm2d(out_channels)
    
        self.tanh = nn.Tanh()
        self.relu = nn.LeakyReLU(LEAKY_ALPHA)

    def forward(self, x, A=None, alpha=1):
        N, C, T, V = x.size()
        res = x
        q, k, v = self.conv1(x).mean(-2), self.conv2(x).mean(-2), self.conv3(x).view(N, self.num_scale, self.Nh, -1, T, V)
        weights = self.conv4(self.tanh(q.unsqueeze(-1) - k.unsqueeze(-2))).view(N, self.num_scale, self.Nh, -1, V, V)        
        weights = weights * self.alpha.to(weights.dtype) + self.A.view(1, 1, self.Nh, 1, V, V).to(weights.dtype)
        x = torch.einsum('ngacvu, ngactu->ngctv', weights, v).contiguous().view(N, -1, T, V)
        x = self.bn(x)
        return x
    
    
class DeSGC(nn.Module):
    '''
    Note: This module is not included in the open-source release due to subsequent research and development. 
    It will be made available in future updates after the completion of related studies.
    '''
    def __init__(self, in_channels, out_channels, A, k, num_scale=4, num_frame=64, num_joint=25):
        super(DeSGC, self).__init__()

        A = torch.from_numpy(A.astype(np.float32))
        self.Nh = A.size(0)
        self.A = nn.Parameter(A)
        self.B = nn.Parameter(A)
        
        self.num_scale = num_scale
        self.k = k
        self.delta = 10
        
        rel_channels = in_channels // 8 if in_channels != 3 else 8
        self.factor = rel_channels // num_scale
        
        self.pe = PositionalEncoding(in_channels, num_joint, num_frame)
        self.conv = PointWiseTCN(in_channels, out_channels * self.Nh, 1, groups=num_scale)
        self.convQK = nn.Conv2d(in_channels, 2 * rel_channels * self.Nh, 1, groups=num_scale)
        self.convW = nn.Conv2d(rel_channels * self.Nh, out_channels * self.Nh, 1, groups=num_scale * self.Nh)

        self.alpha = nn.Parameter(torch.zeros(1))
        self.beta = nn.Parameter(torch.zeros(1, 1, self.Nh, 1, 1, 1))
        self.bn = nn.BatchNorm2d(out_channels)
    
        self.tanh = nn.Tanh()
        self.relu = nn.LeakyReLU(LEAKY_ALPHA)

    def forward(self, x):
        N, C, T, V = x.size()
        res = x
        v = self.relu(self.conv(x)).view(N, self.num_scale, self.Nh, -1, T, V)
        dtype, device = v.dtype, v.device
        
        # calculate score
        # ...
        
        # calculate weight
        # ...
        
        # convert to onehot
        # ...
        
        # sampling & aggregation
        # ...
        
        return x
    

class DeTGC(nn.Module):
    def __init__(self, in_channels, out_channels, eta, kernel_size=1, stride=1, padding=0, dilation=1, 
                 num_scale=1, num_frame=64):
        super(DeTGC, self).__init__()
        
        self.ks, self.stride, self.dilation = kernel_size, stride, dilation
        self.T = num_frame
        self.num_scale = num_scale
        
        self.eta = eta
        ref = (self.ks + (self.ks-1) * (self.dilation-1) - 1) // 2
        tr = torch.linspace(-ref, ref, self.eta)
        self.tr = nn.Parameter(tr)

        self.conv_out = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=(self.eta, 1, 1)),
            nn.BatchNorm3d(out_channels)
        )

    def forward(self, x):
        res = x
        N, C, T, V = x.size()
        Tout = T // self.stride
        dtype = x.dtype 
        
        #learnable sampling locations
        t0 = torch.arange(0, T, self.stride, dtype=dtype, device=x.device)
        tr = self.tr.to(dtype)
        t0, tr = t0.view(1, 1, -1).expand(-1, self.eta, -1), tr.view(1, self.eta, 1) 
        t = t0 + tr 
        t = t.view(1, 1, -1, 1) 
        
        #indexing
        tdn = t.detach().floor()
        tup = tdn + 1
        index1, index2 = torch.clamp(tdn, 0, self.T-1).long(), torch.clamp(tup, 0, self.T-1).long()
        index1, index2 = index1.expand(N, C, -1, V), index2.expand(N, C, -1, V)
        
        #sampling
        alpha = tup - t
        x1, x2 = x.gather(-2, index=index1), x.gather(-2, index=index2) 
        x = x1 * alpha + x2 * (1 - alpha)
        x = x.view(N, C, self.eta, Tout, V)
        
        #conv
        x = self.conv_out(x).squeeze(2)
        return x


class MultiScale_TemporalModeling(nn.Module):
    def __init__(self, in_channels, out_channels, eta, kernel_size=5, stride=1, dilations=1, 
                 num_scale=1, num_frame=64):
        super(MultiScale_TemporalModeling, self).__init__()
        
        scale_channels = out_channels // num_scale
        self.num_scale = num_scale if in_channels !=3 else 1

        self.tcn1 = nn.Sequential(
            PointWiseTCN(in_channels, scale_channels),
            nn.LeakyReLU(LEAKY_ALPHA),
            DeTGC(scale_channels, 
                  scale_channels, 
                  eta,
                  kernel_size=5, 
                  stride=stride, 
                  dilation=1, 
                  num_scale=num_scale, 
                  num_frame=num_frame)
        )
        
        self.tcn2 = nn.Sequential(
            PointWiseTCN(in_channels, scale_channels),
            nn.LeakyReLU(LEAKY_ALPHA),
            DeTGC(scale_channels, 
                  scale_channels, 
                  eta,
                  kernel_size=5, 
                  stride=stride, 
                  dilation=2, 
                  num_scale=num_scale, 
                  num_frame=num_frame)
        )
        
        self.maxpool3x1 = nn.Sequential(
            PointWiseTCN(in_channels, scale_channels),
            nn.LeakyReLU(LEAKY_ALPHA),
            nn.MaxPool2d(kernel_size=(3,1), stride=(stride,1), padding=(1,0)),
            nn.BatchNorm2d(scale_channels) 
        )
        self.conv1x1 = PointWiseTCN(in_channels, scale_channels, stride=stride)

    def forward(self, x):
        x = torch.cat([self.tcn1(x), self.tcn2(x), self.maxpool3x1(x), self.conv1x1(x)], 1)
        return x
    
    
class Basic_Block(nn.Module):
    def __init__(self, in_channels, out_channels, A, k, eta, kernel_size=5, stride=1, dilations=2, 
                 num_frame=64, num_joint=25, residual=True):
        super(Basic_Block, self).__init__()
        
        num_scale = 4
        scale_channels = out_channels // num_scale
        self.num_scale = num_scale if in_channels !=3 else 1
        
        if in_channels == 3:
            self.gcn = ST_GC(in_channels, out_channels, A)
        else:
            # self.gcn = DeSGC(in_channels, 
            #                  out_channels, 
            #                  A, 
            #                  k, 
            #                  self.num_scale, 
            #                  num_frame=num_frame, 
            #                  num_joint=num_joint)
            self.gcn = CTR_GC(in_channels, 
                              out_channels, 
                              A, 
                              self.num_scale)
        self.tcn = MultiScale_TemporalModeling(out_channels, 
                                               out_channels, 
                                               eta,
                                               stride=stride, 
                                               num_scale=num_scale, 
                                               num_frame=num_frame) 
        
        if in_channels != out_channels:
            self.residual1 = PointWiseTCN(in_channels, out_channels, groups=self.num_scale)
        else:
            self.residual1 = lambda x: x
            
        if not residual:
            self.residual2 = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual2 = lambda x: x
        else:
            self.residual2 = PointWiseTCN(in_channels, out_channels, stride=stride, groups=self.num_scale)
        
        self.relu = nn.LeakyReLU(LEAKY_ALPHA)
        init_param(self.modules())
        
    def forward(self, x):
        res = x
        x = self.gcn(x)
        x = self.relu(x + self.residual1(res))
        x = self.tcn(x)
        x = self.relu(x + self.residual2(res))
        return x
    

def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod

def bn_init(bn, scale):
    nn.init.constant_(bn.weight, scale)
    nn.init.constant_(bn.bias, 0)            

    
class DeGCN(nn.Sequential):
    def __init__(self, block_args, A, k, eta):
        super(DeGCN, self).__init__()
        for i, [in_channels, out_channels, stride, residual, num_frame, num_joint] in enumerate(block_args):
            self.add_module(f'block-{i}_tcngcn', Basic_Block(in_channels, 
                                                             out_channels, 
                                                             A, 
                                                             k,
                                                             eta,
                                                             stride=stride, 
                                                             num_frame=num_frame, 
                                                             num_joint=num_joint, 
                                                             residual=residual))  
            

class Model(nn.Module):
    def __init__(self, num_class=60, num_point=25, num_person=2, k=8, eta=4, num_stream=2, 
                 graph=None, graph_args=dict(), in_channels=3, drop_out=0,):
        super(Model, self).__init__()

        if graph is None:
            raise ValueError()
        
        if isinstance(graph, str):
            GraphClass = import_class(graph)
        else:
            GraphClass = graph
        
        self.graph = GraphClass(**graph_args)
        A = self.graph.A

        self.num_class = num_class
        self.num_point = num_point
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * num_point)

        base_channel = 64
        base_frame = 60
        
        self.blockargs = [
            [in_channels, base_channel, 1, False, base_frame, num_point],
            [base_channel, base_channel, 1, True, base_frame, num_point],
            [base_channel, base_channel, 1, True, base_frame, num_point],
            [base_channel, base_channel, 1, True, base_frame, num_point],
            [base_channel, base_channel*2, 2, True, base_frame, num_point],
            [base_channel*2, base_channel*2, 1, True, base_frame//2, num_point],
            [base_channel*2, base_channel*2, 1, True, base_frame//2, num_point],
            [base_channel*2, base_channel*4, 2, True, base_frame//2, num_point],
            [base_channel*4, base_channel*4, 1, True, base_frame//4, num_point],
            [base_channel*4, base_channel*4, 1, True, base_frame//4, num_point]
        ]
        
        self.num_stream = num_stream
        self.streams = nn.ModuleList([DeGCN(self.blockargs, A, k, eta) for _ in range(self.num_stream)])
        self.fc = nn.ModuleList([nn.Linear(base_channel*4, num_class) for _ in range(self.num_stream)])
        
        for fc in self.fc:
            nn.init.normal_(fc.weight, 0, math.sqrt(2. / num_class))
        bn_init(self.data_bn, 1)        
        
        if drop_out:
            self.drop_out = nn.Dropout(drop_out)
        else:
            self.drop_out = lambda x: x
        
    
    def forward(self, x):
        if len(x.shape) == 3:
            N, T, VC = x.shape
            x = x.view(N, T, self.num_point, -1).permute(0, 3, 1, 2).contiguous().unsqueeze(-1)
        N, C, T, V, M = x.size()

        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T).permute(0, 1, 3, 4, 2).contiguous().view(N * M, C, T, V)
        
        x_ = x  
        out = []
        for stream, fc in zip(self.streams, self.fc):
            x = x_
            x = stream(x)
            c_new = x.size(1)
            x = x.view(N, M, c_new, -1)
            x = x.mean(3).mean(1)
            x = self.drop_out(x)
            out.append(fc(x))

        return sum(out) / len(out)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

from tqdm import tqdm

def normalize_skeleton(X, width=1280, height=720):
    """
    X shape: (N, T, M, V, C)
    C = (x, y, confidence)
    """

    X = X.copy()

    # center theo hip (joint 8)
    hip = X[:, :, :, 8:9, :2]
    X[:, :, :, :, :2] = X[:, :, :, :, :2] - hip

    # normalize theo frame size
    X[:, :, :, :, 0] /= width
    X[:, :, :, :, 1] /= height

    # lấy confidence
    conf = X[:, :, :, :, 2:3]

    # confidence weighting
    X[:, :, :, :, 0:1] = X[:, :, :, :, 0:1] * conf
    X[:, :, :, :, 1:2] = X[:, :, :, :, 1:2] * conf

    return X
# ==================================
# DATASET CLASS
# ==================================

class MyDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        x = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.long)

        return x, y, idx


# ==================================
# METRICS (3 CLASS)
# ==================================

def print_metrics(y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred)

    print("\nConfusion Matrix")
    print(cm)

    acc = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="macro"
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro"
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    print("\nMetrics")

    print(f"Accuracy : {acc*100:.2f}%")
    print(f"Precision: {precision*100:.2f}%")
    print(f"Recall   : {recall*100:.2f}%")
    print(f"F1-score : {f1*100:.2f}%")


# ==================================
# PROCESSOR
# ==================================

class Processor:

    def __init__(self, model, arg):

        self.arg = arg

        self.model = model.cuda()

        self.output_device = arg.device if isinstance(arg.device, int) else arg.device[0]

        self.best_acc = 0
        self.best_acc_epoch = 0
        self.start_epoch = arg.start_epoch

        os.makedirs(arg.work_dir, exist_ok=True)

        # ==================================
        # LOAD DATASET
        # ==================================

        print("Loading dataset...")

        # =====================
        # TRAIN
        # =====================
        
        X_train = np.load(
            '../X_train.npy'
        )
        
        y_train = np.load(
            '../y_train.npy'
        )
        
        # =====================
        # VAL
        # =====================
        
        X_val = np.load(
            '../X_val.npy'
        )
        
        y_val = np.load(
            '../y_val.npy'
        )
        
        # =====================
        # TEST
        # =====================
        
        test_X = np.load(
            '../X_test.npy'
        )
        
        test_y = np.load(
            '../y_test.npy'
        )
        print("Normalizing skeleton data...")

        X_train = normalize_skeleton(X_train, 1280, 720)
        X_val   = normalize_skeleton(X_val, 1280, 720)
        test_X  = normalize_skeleton(test_X, 1280, 720)
        # =====================
        # TRANSPOSE
        # (N,T,M,V,C) → (N,C,T,V,M)
        # =====================
        
        X_train = np.transpose(X_train, (0,4,1,3,2))
        X_val   = np.transpose(X_val,   (0,4,1,3,2))
        test_X  = np.transpose(test_X,  (0,4,1,3,2))
        print("Train:", X_train.shape)
        print("Val:", X_val.shape)
        print("Test:", test_X.shape)
        # ==================================
        # DATALOADER
        # ==================================

        self.data_loader = {

            'train': DataLoader(
                MyDataset(X_train, y_train),
                batch_size=arg.batch_size,
                shuffle=True,
                num_workers=2
            ),

            'val': DataLoader(
                MyDataset(X_val, y_val),
                batch_size=arg.test_batch_size,
                shuffle=False,
                num_workers=2
            ),

            'test': DataLoader(
                MyDataset(test_X, test_y),
                batch_size=arg.test_batch_size,
                shuffle=False,
                num_workers=2
            )
        }

        # ==================================
        # LOSS + OPTIMIZER
        # ==================================

        self.loss = nn.CrossEntropyLoss().cuda(self.output_device)

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=arg.base_lr,
            weight_decay=arg.weight_decay
        )

        # ==================================
        # RESUME TRAINING
        # ==================================

        resume_path = os.path.join(
            arg.work_dir,
            "latest_checkpoint.pt"
        )

        if os.path.exists(resume_path):

            checkpoint = torch.load(
                resume_path,
                weights_only=False
            )

            self.model.load_state_dict(
                checkpoint['model_state']
            )

            self.optimizer.load_state_dict(
                checkpoint['optim_state']
            )

            self.best_acc = checkpoint['best_acc']
            self.best_acc_epoch = checkpoint['best_acc_epoch']

            self.start_epoch = checkpoint['epoch'] + 1

            print(
                f"Resumed from epoch {self.start_epoch}"
            )

        else:

            print("Training from scratch")


    # ==================================
    # TRAIN
    # ==================================

    def train(self, epoch):

        self.model.train()

        print(f"\nEpoch {epoch+1} Training")

        loader = self.data_loader['train']

        loss_value = []
        acc_value = []

        for data, label, _ in tqdm(loader, ncols=60):

            data = data.cuda()
            label = label.cuda()

            output = self.model(data)

            loss = self.loss(output, label)

            self.optimizer.zero_grad()

            loss.backward()

            self.optimizer.step()

            _, pred = torch.max(output, 1)

            acc = torch.mean(
                (pred == label).float()
            )

            loss_value.append(loss.item())
            acc_value.append(acc.item())

        print(
            f"Train loss: {np.mean(loss_value):.4f}"
        )

        print(
            f"Train acc : {np.mean(acc_value)*100:.2f}%"
        )


    # ==================================
    # EVAL
    # ==================================

    def eval(self, epoch, mode='val'):

        self.model.eval()

        loader = self.data_loader[mode]

        loss_value = []
        score_frag = []
        label_list = []

        with torch.no_grad():

            for data, label, _ in tqdm(loader, ncols=60):

                data = data.cuda()
                label = label.cuda()

                output = self.model(data)

                loss = self.loss(output, label)

                loss_value.append(loss.item())

                score_frag.append(
                    output.cpu().numpy()
                )

                label_list.append(
                    label.cpu().numpy()
                )

        score = np.concatenate(score_frag)

        label_list = np.concatenate(label_list)

        acc = accuracy_score(
            label_list,
            np.argmax(score, axis=1)
        )

        print(
            f"{mode} loss: {np.mean(loss_value):.4f}"
        )

        print(
            f"{mode} acc : {acc*100:.2f}%"
        )

        if mode == 'val' and acc > self.best_acc:

            self.best_acc = acc
            self.best_acc_epoch = epoch + 1

            torch.save(
                self.model.state_dict(),
                os.path.join(
                    self.arg.work_dir,
                    'best_model.pt'
                )
            )

            print("New best model saved")

        return acc


    # ==================================
    # START TRAINING
    # ==================================

    def start(self):

        for epoch in range(
            self.start_epoch,
            self.arg.num_epoch
        ):

            self.train(epoch)

            self.eval(epoch, 'val')

            torch.save({

                'epoch': epoch,

                'model_state': self.model.state_dict(),

                'optim_state': self.optimizer.state_dict(),

                'best_acc': self.best_acc,

                'best_acc_epoch': self.best_acc_epoch

            },

            os.path.join(
                self.arg.work_dir,
                "latest_checkpoint.pt"
            ))

        print(
            f"\nBest val acc {self.best_acc*100:.2f}% "
            f"at epoch {self.best_acc_epoch}"
        )


    # ==================================
    # TEST BEST MODEL
    # ==================================

    def test_best(self):

        print("\nEvaluating BEST model")

        best_model_path = os.path.join(
            self.arg.work_dir,
            'best_model.pt'
        )

        self.model.load_state_dict(
            torch.load(best_model_path)
        )

        self.model.eval()

        all_labels = []
        all_preds = []

        with torch.no_grad():

            for data, label, _ in self.data_loader['test']:

                data = data.cuda()
                label = label.cuda()

                output = self.model(data)

                _, pred = torch.max(output, 1)

                all_labels.append(
                    label.cpu().numpy()
                )

                all_preds.append(
                    pred.cpu().numpy()
                )

        all_labels = np.concatenate(all_labels)
        all_preds = np.concatenate(all_preds)

        print_metrics(
            all_labels,
            all_preds
        )

In [ ]:
class Args:
    device = 0
    batch_size = 16
    test_batch_size = 32
    base_lr = 0.0001
    weight_decay = 1e-4
    num_epoch = 200
    start_epoch = 0
    save_interval = 5
    model_saved_name = 'my_model'
    work_dir = './results'

arg = Args()

model = Model(
    num_class=7,
    num_point=25,
    num_person=9,
    in_channels=3,
    graph=Graph,
    graph_args={'labeling_mode': 'spatial'}
)
processor = Processor(model, arg)
processor.start()       # → Train + validate
processor.test_best()   # → Load best model và test cuối cùng
